In [2]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [3]:
from langchain_ollama import ChatOllama


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint

In [5]:
@tool
def multiply(a:int, b:int) -> int:
  """ Given two it will multiply them"""
  return a*b

In [6]:
multiply.invoke({"a" : 2 , "b" : 3})

6

In [7]:
multiply.name

'multiply'

In [8]:
multiply.description

'Given two it will multiply them'

In [9]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [10]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0.1
)

In [11]:
llm.invoke("hi Bro")

AIMessage(content='Hey there! How can I assist you today? Feel free to ask me any questions or let me know if you need any help with something.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-13T17:56:23.0353336Z', 'done': True, 'done_reason': 'stop', 'total_duration': 27452223700, 'load_duration': 15910823500, 'prompt_eval_count': 31, 'prompt_eval_duration': 1890441000, 'eval_count': 30, 'eval_duration': 9630559000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a13668-1e2b-7e70-830f-f9419ea68ddb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 30, 'total_tokens': 61})

In [12]:
llm_with_tools = llm.bind_tools([multiply])

In [13]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm just a digital assistant, so I don't have feelings, but I'm here and ready to help you with any questions or tasks you might have. How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-13T17:56:47.4945273Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22384072400, 'load_duration': 362511300, 'prompt_eval_count': 154, 'prompt_eval_duration': 6934279000, 'eval_count': 42, 'eval_duration': 15063009000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a13668-9193-7302-a0a2-a7ef93040626-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 154, 'output_tokens': 42, 'total_tokens': 196})

In [20]:
query = HumanMessage("Can you multiply 100 with 989")

In [21]:
messages = [query]

In [22]:
messages


[HumanMessage(content='Can you multiply 100 with 989', additional_kwargs={}, response_metadata={})]

In [23]:
result = llm_with_tools.invoke(messages)

In [24]:
messages.append(result)

In [25]:
messages


[HumanMessage(content='Can you multiply 100 with 989', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-13T18:14:00.0416936Z', 'done': True, 'done_reason': 'stop', 'total_duration': 37213008600, 'load_duration': 15833560600, 'prompt_eval_count': 162, 'prompt_eval_duration': 10882903000, 'eval_count': 29, 'eval_duration': 10473869000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a13678-1901-7e31-bc64-6f7401b86bec-0', tool_calls=[{'name': 'multiply', 'args': {'a': 100, 'b': 989}, 'id': 'dfdad84b-37d8-4cc4-a2fa-47ba74792044', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 29, 'total_tokens': 191})]

In [26]:
tool_result = multiply.invoke(result.tool_calls[0])

In [27]:
tool_result

ToolMessage(content='98900', name='multiply', tool_call_id='dfdad84b-37d8-4cc4-a2fa-47ba74792044')

In [31]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 100, 'b': 989},
  'id': 'dfdad84b-37d8-4cc4-a2fa-47ba74792044',
  'type': 'tool_call'}]

In [33]:
messages.append([tool_result])

In [34]:
messages

[HumanMessage(content='Can you multiply 100 with 989', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-13T18:14:00.0416936Z', 'done': True, 'done_reason': 'stop', 'total_duration': 37213008600, 'load_duration': 15833560600, 'prompt_eval_count': 162, 'prompt_eval_duration': 10882903000, 'eval_count': 29, 'eval_duration': 10473869000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a13678-1901-7e31-bc64-6f7401b86bec-0', tool_calls=[{'name': 'multiply', 'args': {'a': 100, 'b': 989}, 'id': 'dfdad84b-37d8-4cc4-a2fa-47ba74792044', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 29, 'total_tokens': 191}),
 [ToolMessage(content='98900', name='multiply', tool_call_id='dfdad84b-37d8-4cc4-a2fa-47ba74792044')]]

In [41]:
llm_with_tools.invoke(messages[0]).content

ValueError: Invalid input type <class 'langchain_core.messages.human.HumanMessage'>. Must be a PromptValue, str, or list of BaseMessages.

In [51]:
# tool create 
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [ ]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fectches the currency conversion factor between a given base currency and a target currency
    """


    api_key = "a18d6f6a64c3dcd36d3a70e0"
    base_currency = "USD"
    target_currency = "INR"

    url = (
     f"https://v6.exchangerate-api.com/v6/"
     f"{api_key}/pair/{base_currency}/{target_currency}"
     )
    response = requests.get(url, timeout=15)

    print(response)

    response.raise_for_status()

    if not response.text.strip():

     raise ValueError("The API returned an empty response.")

    try:
     data = response.json()
    except requests.exceptions.JSONDecodeError:
     raise ValueError(
        f"The API did not return valid JSON: {response.text[:300]}"
     )

    if data.get("result") != "success":
     raise ValueError(f"ExchangeRate-API error: {data}")

    return data 

    # return response.json()

In [120]:
@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
    """
    given a currencyconversion rate this function calculates the target currency value from a given currency
    """

    return base_currency_value * conversion_rate

In [113]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [116]:
import requests

api_key = "a18d6f6a64c3dcd36d3a70e0"
base_currency = "USD"
target_currency = "INR"

url = (
    f"https://v6.exchangerate-api.com/v6/"
    f"{api_key}/pair/{base_currency}/{target_currency}"
)

response = requests.get(url, timeout=15)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response body:", repr(response.text[:1000]))

Status code: 200
Content type: application/json
Response body: '{"result":"success","documentation":"https://www.exchangerate-api.com/docs","terms_of_use":"https://www.exchangerate-api.com/terms","time_last_update_unix":1791849601,"time_last_update_utc":"Tue, 13 Oct 2026 00:00:01 +0000","time_next_update_unix":1791936001,"time_next_update_utc":"Wed, 14 Oct 2026 00:00:01 +0000","base_code":"USD","target_code":"INR","conversion_rate":95.9069}'


In [121]:
get_conversion_factor.invoke({'base_currency': 'USD','target_currency':'INR'})


<Response [200]>


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791849601,
 'time_last_update_utc': 'Tue, 13 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791936001,
 'time_next_update_utc': 'Wed, 14 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.9069}

In [122]:
convert.invoke({'base_currency_value':10, 'conversion_rate' : 96})

960.0

In [123]:
from langchain_ollama import ChatOllama

In [124]:
llm = ChatOllama(
    model="qwen2.5:7b"
)

In [125]:
# Tool Binding
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [141]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that you convert 89 INR to USD and give me the output')]

In [127]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that you convert 89 INR to USD', additional_kwargs={}, response_metadata={})]

In [128]:
ai_message = llm_with_tools.invoke(messages)

In [129]:
messages.append(ai_message)

In [130]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc34bda8-c95c-4de0-b24c-188de2bb2469',
  'type': 'tool_call'}]

In [137]:
import json

for tool_call in ai_message.tool_calls:
# Execute the first tool and get me the Conversation rate

    if tool_call['name'] == 'get_conversion_factor':
    # Below is the code for fetching the conversion rate
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversation_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to message list

        messages.append(tool_message1)
    # Execute the 2nd tool using the conversion rate form tool1
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        message.append(tool_message2)



<Response [200]>


In [138]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that you convert 89 INR to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-13T19:09:53.5098906Z', 'done': True, 'done_reason': 'stop', 'total_duration': 31301847900, 'load_duration': 14297527100, 'prompt_eval_count': 252, 'prompt_eval_duration': 10168238000, 'eval_count': 30, 'eval_duration': 6767673000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a136ab-5b9c-7640-a1ad-64841338349a-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'fc34bda8-c95c-4de0-b24c-188de2bb2469', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 252, 'output_tokens': 30, 'total_tokens': 282}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-

In [143]:
llm_with_tools.invoke(messages).content

''

In [147]:
from langchain.agents import creat_agent
agent_executor = initialize_agent(
    tools=[get_conversion_factor,convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

ImportError: cannot import name 'creat_agent' from 'langchain.agents' (c:\Users\cbl7413\OneDrive - Takeda\Desktop\Langchain\.venv\Lib\site-packages\langchain\agents\__init__.py)

In [146]:
import langchain
print(langchain.__version__)

1.4.3


In [148]:
print("Hello World ")

Hello World 


In [149]:
print("Hello world")

Hello world
